In [20]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'
with h5py.File(filename, 'r') as f:
    #low_cat_matrix = f['data/lowest_tier_category_matrix_raw'][:]
    high_cat_matrix = f['data/highest_tier_category_matrix_raw'][:] 
    gene_names = f['meta/genes/all_gene_names'][:]
    gene_description = f['meta/genes/gene_description'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
gene_names = [x.decode('UTF-8') for x in gene_names]
gene_description = [x.decode('UTF-8') for x in gene_description]

category_list  = ["Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]

category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

In [17]:
low_cat_matrix[0][0:3]

array([219.33996383, 110.86      ,          nan])

In [24]:
high_cat_matrix[0][0:3]

array([ 94.79237845,  99.22251054, 113.55603686])

In [2]:
from IPython.display import display

df = pd.DataFrame(low_cat_matrix, index=gene_names)
df.columns = category_list
#display(df)

---
# Calculations

In [3]:

# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(low_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
low_cat_oe_matrix = np.divide(low_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(low_cat_matrix), where=row_averages[:, np.newaxis]!=0)

print("Low Cat matrix shape:")
print(low_cat_matrix.shape)
print(low_cat_matrix[0][:5])
print()
print("Average:", row_averages[0])
print()
print("Low Cat OE matrix shape:")
print(low_cat_oe_matrix.shape)
print(low_cat_oe_matrix[0][:5])

Low Cat matrix shape:
(57447, 127)
[219.33996383 110.86                nan 345.125      138.42222222]

Average: 239.42489560022628

Low Cat OE matrix shape:
(57447, 127)
[0.91611177 0.4630262         nan 1.44147499 0.57814465]


In [4]:
low_cat_log2_oe_matrix = np.log2(low_cat_oe_matrix)

/tmp/ipykernel_7604/140735222.py:1: RuntimeWarning: divide by zero encountered in log2
  low_cat_log2_oe_matrix = np.log2(low_cat_oe_matrix)


In [18]:
from IPython.display import display

df = pd.DataFrame(low_cat_log2_oe_matrix, index=gene_names)
df.columns = category_list
display(df)

,Heart,Atrium,Cardiac Muscle Fiber,Pericardium,Valve,Ventricle,HUVEC,Adipose,Adipocyte,Myofibroblast,...,C42,DU145,LNCAP,PC3,Ovary,Granulosa,Oocyte,ES2,SKOV3,Testis
A1BG,-0.126404,-1.110834,NaN,0.527546,-0.790498,-2.200903,-2.446766,-1.542154,-1.591312,NaN,...,-5.816595,-4.064384,-0.606466,-0.712436,0.535239,-1.685739,-2.939857,0.182658,-0.133412,0.468637
A1BG-AS1,0.598463,0.179362,NaN,-0.884835,-0.028288,0.233744,0.397010,0.259864,-4.448816,NaN,...,-1.978446,-1.308816,0.750940,0.976939,3.176205,1.497126,-0.814314,2.754381,2.404521,2.295755
A1CF,-4.228363,-3.286436,NaN,-2.725962,-4.733680,-6.339132,-7.094095,-6.547884,-6.206574,NaN,...,-11.619197,-10.824222,-6.092433,-5.983414,-6.990661,-8.457205,-8.809394,-5.201510,-5.689734,-5.817281
A2M,0.153984,1.204774,NaN,-3.361104,0.875401,1.041917,-0.447563,0.383343,-4.522323,NaN,...,-9.729420,-2.218333,-1.779542,-3.595216,-1.589418,-4.985718,-6.999131,-1.687847,-3.252751,-2.115025
A2M-AS1,1.440372,2.288570,NaN,-0.321520,1.593364,1.465283,0.096950,1.476748,3.874589,NaN,...,-8.150408,-1.113268,-0.542241,-1.135465,-2.132396,-3.761219,-3.997547,-1.078914,-1.209054,-0.176152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11B,1.682617,2.875920,NaN,-1.235624,2.418438,1.896681,1.385733,1.029639,-2.439324,NaN,...,-5.923302,-1.962767,0.215054,1.410810,2.024401,0.311653,-1.509874,2.140166,1.511719,2.461245
ZYX,0.601410,-0.240556,NaN,-1.769472,-0.314893,-0.059928,2.140431,0.928924,-6.881899,NaN,...,-4.736717,-1.506200,0.613965,2.340994,2.953125,1.151157,-0.679192,2.643050,1.644030,2.170126
ZYXP1,1.388560,-inf,NaN,-inf,-inf,-inf,0.235253,0.443811,-inf,NaN,...,-inf,-inf,3.117913,0.955376,-inf,-inf,-inf,-inf,-inf,-inf
ZZEF1,1.064221,1.911874,NaN,-1.527900,1.208248,0.940695,1.307793,0.898586,-4.049425,NaN,...,-5.169541,-1.288273,0.423803,1.645842,1.692691,-0.233825,-1.701407,1.729176,1.032170,1.451890


---
# Results

In [6]:
df = pd.DataFrame(low_cat_matrix, index=gene_names)
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Heart":
TTN       647856.631103
MALAT1    511859.195298
RN7SL2    260951.490054
MYH7      255844.972875
RN7SL1    145478.508137
FN1       135605.459313
EEF1A1    102114.567812
DES        92970.265823
TPM1       87134.278481
ACTC1      84454.625678
Name: Heart, dtype: float64

Gene descriptions for Column "Heart":
titin
metastasis associated lung adenocarcinoma transcript 1
RNA component of signal recognition particle 7SL2
myosin heavy chain 7
RNA component of signal recognition particle 7SL1
fibronectin 1
eukaryotic translation elongation factor 1 alpha 1
desmin
tropomyosin 1
actin alpha cardiac muscle 1

Top 10 values in Column "Atrium":
TTN       2524683.58
RN7SL2    1031456.58
RN7SL1     664775.54
MYH7       529564.36
HBB        411835.96
DES        197321.08
HBA1       178785.18
HBG2       152931.90
TPM1       139918.16
MALAT1     135754.94
Name: Atrium, dtype: float64

Gene descriptions for Column "Atrium":
titin
RNA component of signal recognition particl

Top 10 values in Column "Liver":
ENSG00000203396    67330.626384
EEF1A1             63764.059562
MALAT1             29928.309623
B2M                27827.704159
RPL7               24778.903027
IGKC               23750.843958
HNRNPH1            20487.428009
RPL3               19652.508737
HBB                17878.974895
CD74               16047.657150
Name: Liver, dtype: float64

Gene descriptions for Column "Liver":
WDR45-like (WDR45L) pseudogene
eukaryotic translation elongation factor 1 alpha 1
metastasis associated lung adenocarcinoma transcript 1
beta-2-microglobulin
ribosomal protein L7
immunoglobulin kappa constant
heterogeneous nuclear ribonucleoprotein H1
ribosomal protein L3
hemoglobin subunit beta
CD74 molecule

Top 10 values in Column "Hepatic Stellate Cell":
EEF1A1    121512.564356
LYZ        48985.257426
ACTB       36307.168317
TPT1       29258.455446
MPO        27831.603960
RPL3       27782.693069
B2M        26325.346535
S100A9     26185.396040
GAPDH      22340.346535
RPS

Top 10 values in Column "Macrophage":
EEF1A1             200626.407855
MALAT1             151555.270695
ALB                145829.383988
RN7SL1             136053.630816
RN7SL2             118421.233535
ACTB                76354.265257
IGKC                62978.683082
ENSG00000282885     60114.814502
ACTG1               57247.224773
GAPDH               54840.466767
Name: Macrophage, dtype: float64

Gene descriptions for Column "Macrophage":
eukaryotic translation elongation factor 1 alpha 1
metastasis associated lung adenocarcinoma transcript 1
albumin
RNA component of signal recognition particle 7SL1
RNA component of signal recognition particle 7SL2
actin beta
immunoglobulin kappa constant
novel transcript
actin gamma 1
glyceraldehyde-3-phosphate dehydrogenase

Top 10 values in Column "Macroglia":
A1BG        NaN
A1BG-AS1    NaN
A1CF        NaN
A2M         NaN
A2M-AS1     NaN
A2ML1       NaN
A2ML1-AS1   NaN
A2ML1-AS2   NaN
A2MP1       NaN
A3GALT2     NaN
Name: Macroglia, dtype: float6

Top 10 values in Column "Vascular Smooth Muscle":
RN7SKP255    28517.935484
FTL          10813.338710
APOA2         7464.354839
ACTB          6743.532258
GAPDH         6046.967742
FGG           5582.580645
RPL3          5369.758065
AFP           4329.370968
RN7SL674P     4227.306452
BSG           4015.919355
Name: Vascular Smooth Muscle, dtype: float64

Gene descriptions for Column "Vascular Smooth Muscle":
RN7SK pseudogene 255
ferritin light chain
apolipoprotein A2
actin beta
glyceraldehyde-3-phosphate dehydrogenase
fibrinogen gamma chain
ribosomal protein L3
alpha fetoprotein
RNA, 7SL, cytoplasmic 674, pseudogene
basigin (Ok blood group)

Top 10 values in Column "LHCNM2":
ALB          66168.05
FTL          52042.95
EEF1A1       48945.65
RN7SKP255    30079.60
AFP          29344.05
APOB         29320.65
FN1          27541.40
SERPINA1     27517.30
APOA2        22234.30
TF           19513.40
Name: LHCNM2, dtype: float64

Gene descriptions for Column "LHCNM2":
albumin
ferritin light chain

Top 10 values in Column "Motor Neuron":
GCG                101811.481707
MALAT1              39511.856707
ENSG00000225840     27315.868902
TTR                 25731.240854
ENSG00000280614     22813.606707
ENSG00000280800     22813.606707
ENSG00000281181     22813.606707
INS                 20441.884146
EEF1A1               5685.189024
B2M                  4817.207317
Name: Motor Neuron, dtype: float64

Gene descriptions for Column "Motor Neuron":
glucagon
metastasis associated lung adenocarcinoma transcript 1
RNA, 18S ribosomal pseudogene
transthyretin
novel transcript, similar to YY1 associated myogenesis RNA 1 YAM1
novel transcript, similar to YY1 associated myogenesis RNA 1 YAM1
novel transcript, similar to YY1 associated myogenesis RNA 1 YAM1
insulin
eukaryotic translation elongation factor 1 alpha 1
beta-2-microglobulin

Top 10 values in Column "Sensory Neuron":
GCG                139978.854839
ENSG00000280614     44867.338710
ENSG00000280800     44867.338710
ENSG00000281181     4

Top 10 values in Column "HELA":
EEF1A1             131666.240118
FN1                114269.388992
RN7SL2              85784.115257
RN7SL1              85389.580717
MALAT1              81222.224603
COL1A1              60337.022165
COL1A2              58006.020687
ACTB                51257.612856
ENSG00000283907     42925.664573
THBS1               38387.044699
Name: HELA, dtype: float64

Gene descriptions for Column "HELA":
eukaryotic translation elongation factor 1 alpha 1
fibronectin 1
RNA component of signal recognition particle 7SL2
RNA component of signal recognition particle 7SL1
metastasis associated lung adenocarcinoma transcript 1
collagen type I alpha 1 chain
collagen type I alpha 2 chain
actin beta
novel transcript, antisense to ATP4A
thrombospondin 1

Top 10 values in Column "Kidney":
KRT10     137371.164338
KRT1      133311.874848
EEF1A1    132484.310753
KRT14      85744.675577
KRT5       78670.144289
RN7SL2     72418.713244
RN7SL1     64284.028250
MALAT1     57105.428919
K

In [7]:
df = pd.DataFrame(low_cat_oe_matrix, index=gene_names)
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Heart":
RNU1-61P           60.186601
ENSG00000274866    56.043351
ENSG00000239628    51.691509
ENSG00000237324    45.190871
ENSG00000253805    44.802203
ENSG00000286639    44.258963
ENSG00000252868    43.734715
MYH1               41.970089
RNA5SP162          41.964429
GJD2-DT            41.900047
Name: Heart, dtype: float64

Gene descriptions for Column "Heart":
RNA, U1 small nuclear 61, pseudogene
None
novel transcript
novel transcript
novel transcript
novel transcript
None
myosin heavy chain 1
RNA, 5S ribosomal pseudogene 162
GJD2 divergent transcript

Top 10 values in Column "Atrium":
MIR7151            96.599024
MIR4463            95.961860
MIR4780            95.349347
RNU6-699P          92.684612
MIR4493            88.980072
ENSG00000201666    86.250184
ENSG00000230112    84.542970
RNA5SP501          84.198270
RNU1-34P           83.329044
RNU7-25P           82.983795
Name: Atrium, dtype: float64

Gene descriptions for Column "Atrium":
microRNA 7151
microRN

Top 10 values in Column "NHDF":
ENSG00000266196    42.751082
ENSG00000216663    30.736561
LINC02729           7.180965
ST13P15             7.074683
ENSG00000273362     4.954540
YBX1P6              4.574809
LPAR4               4.254109
RPL12P13            3.221031
ENSG00000273300     3.171361
LRRC38              2.971401
Name: NHDF, dtype: float64

Gene descriptions for Column "NHDF":
novel transcript
hepatocellular carcinoma-related putative tumor suppressor (LPTS) pseudogene
long intergenic non-protein coding RNA 2729
ST13, Hsp70 interacting protein pseudogene 15
novel transcript
Y-box binding protein 1 pseudogene 6
lysophosphatidic acid receptor 4
ribosomal protein L12 pseudogene 13
novel transcript, antisense to UFD1L
leucine rich repeat containing 38

Top 10 values in Column "HCT116":
TSPY7P         101.578513
IGHV3OR16-6     98.205973
SNORD18B        88.355201
MIR124-1        82.246019
MIR1244-1       75.175425
MIR1244-2       75.175425
MIR1244-3       75.175425
MIR1244-4       75

Top 10 values in Column "PANC1":
ENSG00000277468    99.473839
MIR5704            88.255554
CYCSP26            86.513358
LINC01304          77.088659
ENSG00000260335    76.263949
LINC02633          75.943092
OR1X5P             74.124775
TMPOP1             69.819554
ENSG00000278650    68.423508
MIR5090            67.755680
Name: PANC1, dtype: float64

Gene descriptions for Column "PANC1":
None
microRNA 5704
CYCS pseudogene 26
long intergenic non-protein coding RNA 1304
coronin, actin binding protein, 1A (CORO1A) pseudogene
long intergenic non-protein coding RNA 2633
olfactory receptor family 1 subfamily X member 5 pseudogene
thymopoietin pseudogene 1
adenosine deaminase domain containing 1 (testis-specific) (ADAD1) pseudogene
microRNA 5090

Top 10 values in Column "Gastric Epithelial Cell":
ENSG00000253638    98.133714
TRIM60P11Y         91.661164
ENSG00000258615    90.302783
HIGD1AP10          90.171734
TRIM60P9Y          89.789060
ARL2BPP6           87.119182
DLGAP5P1           85.7482

Top 10 values in Column "Macroglia":
ENSG00000239255    0.0
ENSG00000255972    0.0
ENSG00000266416    0.0
ENSG00000271041    0.0
ENSG00000271154    0.0
ENSG00000271544    0.0
ENSG00000282732    0.0
IGHD1-1            0.0
IGHD1-14           0.0
IGHD1-20           0.0
Name: Macroglia, dtype: float64

Gene descriptions for Column "Macroglia":
pseudogene similar to part of guanine nucleotide binding protein (G protein), beta polypeptide 1GNB1
chemokine (C-C motif) receptor-like 2 (CCRL2) pseudogene
lectin, galactoside-binding, soluble, 9 (LGALS9) pseudogene
coiled-coil domain containing 58 (CCDC58) pseudogene
coiled-coil domain containing 58 (CCDC58) pseudogene
myxovirus (influenza virus) resistance 1, interferon-inducible protein p78 (mouse) (MX1) pseudogene
novel pseudogene similar to part of zinc finger protein
immunoglobulin heavy diversity 1-1
immunoglobulin heavy diversity 1-14 (non-functional)
immunoglobulin heavy diversity 1-20

Top 10 values in Column "THP1":
MIR122             10

Top 10 values in Column "IMR32":
RNY1P1             64.298968
RNU6-1220P         51.046448
ENSG00000235679    49.908080
ENSG00000261350    45.306424
ENSG00000238223    37.329730
MTND6P13           34.138511
OR1D4              33.612213
ENSG00000206776    25.350052
ENSG00000253060    23.840088
ENSG00000271589    23.263031
Name: IMR32, dtype: float64

Gene descriptions for Column "IMR32":
RNY1 pseudogene 1
RNA, U6 small nuclear 1220, pseudogene
pseudogene similar to part of SIN3 homolog A, transcriptional regulator (yeast) (SIN3A)
novel transcript
zinc finger protein 33a (ZNF33A) pseudogene
MT-ND6 pseudogene 13
olfactory receptor family 1 subfamily D member 4
None
None
NLR family, pyrin domain containing 3 (NLRP3) pseudogene

Top 10 values in Column "SKNSH":
ENSG00000226905    86.790455
ENSG00000232167    68.477873
RNU6-1334P         43.710446
RNU6-192P          43.233583
ENSG00000275387    30.533251
ENSG00000271022    29.026700
OR2B3              26.573819
TP53TG3GP          25.582830
E

Top 10 values in Column "Trachea":
IGKV2D-30          103.098739
IGKV2-30           101.158114
RNU6-220P           93.242803
ENSG00000255944     86.631676
ENSG00000275448     81.623630
ENSG00000261702     81.584326
RNA5SP262           76.993103
IGKJ2               76.422671
ENSG00000226096     75.716586
RNU6-993P           72.081982
Name: Trachea, dtype: float64

Gene descriptions for Column "Trachea":
immunoglobulin kappa variable 2D-30
immunoglobulin kappa variable 2-30
RNA, U6 small nuclear 220, pseudogene
novel transcript
H3 histone, family 3B (H3.3B) (H3F3B) pseudogene
ribosomal protein S15 (RPS15A) pseudogene
RNA, 5S ribosomal pseudogene 262
immunoglobulin kappa joining 2
contactin associated protein-like 3B (CNTNAP3B) pseudogene
RNA, U6 small nuclear 993, pseudogene

Top 10 values in Column "Breast":
ENSG00000250555    117.000000
IFNA17             109.532212
IFNA10             108.811855
IFNA16             108.742896
IFNA7              108.077926
IFNA4              106.003353
I

Top 10 values in Column "ES2":
NEUROG2            115.657313
OR2AT2P             83.155929
VENTXP2             81.518534
ENSG00000259932     70.750369
ENSG00000260035     67.766967
ENSG00000255860     60.351137
ENSG00000248627     60.292143
TAC3                55.821387
SNORD114-20         53.481424
RN7SL836P           52.055762
Name: ES2, dtype: float64

Gene descriptions for Column "ES2":
neurogenin 2
olfactory receptor family 2 subfamily AT member 2 pseudogene
VENT homeobox pseudogene 2
novel transcript
novel transcript
folate receptor 3 (gamma) (FOLR3) pseudogene
novel transcript
tachykinin precursor 3
small nucleolar RNA, C/D box 114-20
RNA, 7SL, cytoplasmic 836, pseudogene

Top 10 values in Column "SKOV3":
CDY15P             107.686829
RNA5SP135           93.497023
ENSG00000271660     78.553845
ENSG00000277211     74.222958
ENSG00000229755     70.135252
RNU6-374P           68.910392
ENSG00000268582     68.810887
ENSG00000276685     66.544343
LINC01783           65.959545
KRTAP25-

In [8]:
df = pd.DataFrame(low_cat_log2_oe_matrix, index=gene_names)
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Heart":
RNU1-61P           5.911370
ENSG00000274866    5.808471
ENSG00000239628    5.691855
ENSG00000237324    5.497959
ENSG00000253805    5.485498
ENSG00000286639    5.467898
ENSG00000252868    5.450707
MYH1               5.391290
RNA5SP162          5.391095
GJD2-DT            5.388880
Name: Heart, dtype: float64

Gene descriptions for Column "Heart":
RNA, U1 small nuclear 61, pseudogene
None
novel transcript
novel transcript
novel transcript
novel transcript
None
myosin heavy chain 1
RNA, 5S ribosomal pseudogene 162
GJD2 divergent transcript

Top 10 values in Column "Atrium":
MIR7151            6.593937
MIR4463            6.584389
MIR4780            6.575151
RNU6-699P          6.534258
MIR4493            6.475410
ENSG00000201666    6.430456
ENSG00000230112    6.401613
RNA5SP501          6.395719
RNU1-34P           6.380748
RNU7-25P           6.374758
Name: Atrium, dtype: float64

Gene descriptions for Column "Atrium":
microRNA 7151
microRNA 4463
microRNA 4780

Top 10 values in Column "NHDF":
ENSG00000266196    5.417889
ENSG00000216663    4.941884
LINC02729          2.844178
ST13P15            2.822665
ENSG00000273362    2.308751
YBX1P6             2.193712
LPAR4              2.088857
RPL12P13           1.687523
ENSG00000273300    1.665102
LRRC38             1.571143
Name: NHDF, dtype: float64

Gene descriptions for Column "NHDF":
novel transcript
hepatocellular carcinoma-related putative tumor suppressor (LPTS) pseudogene
long intergenic non-protein coding RNA 2729
ST13, Hsp70 interacting protein pseudogene 15
novel transcript
Y-box binding protein 1 pseudogene 6
lysophosphatidic acid receptor 4
ribosomal protein L12 pseudogene 13
novel transcript, antisense to UFD1L
leucine rich repeat containing 38

Top 10 values in Column "HCT116":
TSPY7P         6.666451
IGHV3OR16-6    6.617739
SNORD18B       6.465243
MIR124-1       6.361874
MIR1244-1      6.232189
MIR1244-2      6.232189
MIR1244-3      6.232189
MIR1244-4      6.232189
IGHV1OR21-1    6.1

Top 10 values in Column "Granulocyte":
ENSG00000229732    6.532222
SSU72P5            6.307293
SSU72P6            6.223678
SSU72P2            6.211887
ENSG00000229012    6.097301
MUC22              5.739982
LINC00682          5.593134
OR7C2              5.589685
PHOX2B-AS1         5.552891
PADI3              5.534048
Name: Granulocyte, dtype: float64

Gene descriptions for Column "Granulocyte":
novel transcript
SSU72 pseudogene 5
SSU72 pseudogene 6
SSU72 pseudogene 2
novel transcript
mucin 22
long intergenic non-protein coding RNA 682
olfactory receptor family 7 subfamily C member 2
PHOX2B antisense RNA 1
peptidyl arginine deiminase 3

Top 10 values in Column "Neutrophil":
RNU6-582P          6.184856
ENSG00000255830    5.321742
RBM22P1            5.005237
RNU1-76P           4.999228
RNU6ATAC37P        4.969923
ENSG00000240568    4.965693
ENSG00000286465    4.887636
ENSG00000270234    4.773532
ENSG00000266938    4.762251
IGHV4OR15-8        4.744667
Name: Neutrophil, dtype: float64

Gene

Top 10 values in Column "Thymus":
ENSG00000224127    6.357498
ENSG00000224149    6.338150
ENSG00000284779    5.996258
LHX8               5.892237
ENSG00000261213    5.393565
CYP1A2             5.096387
IGFALS             4.992585
LIPF               4.877776
CYP2E1             4.730133
ENSG00000223991    4.641156
Name: Thymus, dtype: float64

Gene descriptions for Column "Thymus":
novel transcript, antisense to LHX8
novel transcript, antisense to LHX8
novel protein
LIM homeobox 8
novel transcript
cytochrome P450 family 1 subfamily A member 2
insulin like growth factor binding protein acid labile subunit
lipase F, gastric type
cytochrome P450 family 2 subfamily E member 1
novel transcript

Top 10 values in Column "Thymocyte":
ENSG00000270891    5.905530
ENSG00000279440    5.490819
ENSG00000203462    5.342705
ENSG00000267628    4.537319
SAA3P              4.268740
ENSG00000285513    4.180749
ENSG00000251074    3.881828
SAA2               3.842069
ENSG00000255845    3.770249
ENSG0000022399

Top 10 values in Column "Skeletal Muscle":
CSGALNACT2P2       6.239805
HNRNPCL2           5.912853
ENSG00000253215    5.782614
UBE2NP1            5.732392
YTHDF2P1           5.579110
HNRNPCL4           5.469544
BNIP3P6            5.401382
YWHAEP1            5.389309
CBX3P9             5.372872
PRPF4BP1           5.322454
Name: Skeletal Muscle, dtype: float64

Gene descriptions for Column "Skeletal Muscle":
CSGALNACT2 pseudogene 2
heterogeneous nuclear ribonucleoprotein C like 2
novel transcript
ubiquitin conjugating enzyme E2 N pseudogene 1
YTH domain family member 2 pseudogene 1
heterogeneous nuclear ribonucleoprotein C like 4
BCL2 interacting protein 3 pseudogene 6
tyrosine 3-monooxygenase/tryptophan 5-monooxygenase activation protein epsilon pseudogene 1
chromobox 3 pseudogene 9
PRPF4B pseudogene 1

Top 10 values in Column "Myoblast":
ENSG00000237875    6.220972
MIR1225            6.187293
ENSG00000275146    5.938881
RNA5SP370          5.845955
RNA5SP226          5.781449
DUX4L12   

Top 10 values in Column "Cerebral Cortex":
ENSG00000251510    5.205977
ENSG00000261229    4.324477
LINC00588          4.259640
ANKRD36P1          4.200018
ENSG00000254260    4.179475
OR51N1P            4.170223
ENSG00000215310    4.078313
HMGB1P47           4.076938
VSTM2A-OT1         4.076753
ENSG00000233157    4.069209
Name: Cerebral Cortex, dtype: float64

Gene descriptions for Column "Cerebral Cortex":
novel transcript
novel transcript
long intergenic non-protein coding RNA 588
ankyrin repeat domain 36 pseudogene 1
novel transcript
olfactory receptor family 51 subfamily N member 1 pseudogene
lysosomal-associated membrane protein 1 pseudogene
high mobility group box 1 pseudogene 47
VSTM2A overlapping transcript 1
ribosomal protein L4 (RPL4) pseudogene

Top 10 values in Column "Hypothalamus":
NDUFA5P3           5.188715
ENSG00000227131    5.139626
ENSG00000273177    5.116329
ENSG00000232042    4.959070
LINC02779          4.891604
ENSG00000251294    4.880604
OR13C1P            4.73820

Top 10 values in Column "Lung Epithelial":
AMYP1              6.643901
ENSG00000285959    6.273840
AMY2A              6.208454
CELA2A             6.186035
PNLIP              6.043618
CPB1               6.004383
CELA2B             5.985473
PRSS1              5.978024
CELA3B             5.947837
MIR3156-1          5.939611
Name: Lung Epithelial, dtype: float64

Gene descriptions for Column "Lung Epithelial":
amylase alpha pseudogene 1
novel transcript
amylase alpha 2A
chymotrypsin like elastase 2A
pancreatic lipase
carboxypeptidase B1
chymotrypsin like elastase 2B
serine protease 1
chymotrypsin like elastase 3B
microRNA 3156-1

Top 10 values in Column "A549":
MIR20B             6.752427
MIR7108            6.335416
MIR20A             6.266050
MIR614             6.253044
MIR18A             6.072716
ENSG00000288615    5.896010
RNA5SP358          5.887513
MTCYBP23           5.556647
RNU1-136P          5.549403
MIR196B            5.537859
Name: A549, dtype: float64

Gene descriptions for Colu

Top 10 values in Column "MDAMB231":
HMGB1P22           6.766598
RNU2-38P           6.519781
IGHV1OR16-2        6.361464
LGALS16            6.354702
GPR32              6.276887
MIR935             6.127211
CGB5               6.088425
RBMY1J             6.037488
COX8C              6.001294
ENSG00000264281    5.989662
Name: MDAMB231, dtype: float64

Gene descriptions for Column "MDAMB231":
high mobility group box 1 pseudogene 22
RNA, U2 small nuclear 38, pseudogene
immunoglobulin heavy variable 1/OR16-2 (pseudogene)
galectin 16
G protein-coupled receptor 32
microRNA 935
chorionic gonadotropin subunit beta 5
RNA binding motif protein Y-linked family 1 member J
cytochrome c oxidase subunit 8C
ribosomal protein L41 (RPL41) pseudogene

Top 10 values in Column "MDAMB468":
ENSG00000232183    6.537836
ENSG00000286740    6.003957
ENSG00000276428    5.882908
ENSG00000249686    5.763440
ENSG00000286709    5.668719
ENSG00000248739    5.617822
ENSG00000271248    5.582414
ENSG00000236606    5.571586
RN

Top 10 values in Column "DU145":
RNU6-1193P         5.815225
RNU6-1269P         5.815225
RNU6-368P          5.815225
RNU6-538P          5.815225
OR5H6              5.749998
LCE3E              5.582094
LCE3D              5.531482
KRT84              5.517925
ENSG00000265671    5.350154
ENSG00000233703    5.311220
Name: DU145, dtype: float64

Gene descriptions for Column "DU145":
RNA, U6 small nuclear 1193, pseudogene
RNA, U6 small nuclear 1269, pseudogene
RNA, U6 small nuclear 368, pseudogene
RNA, U6 small nuclear 538, pseudogene
olfactory receptor family 5 subfamily H member 6
late cornified envelope 3E
late cornified envelope 3D
keratin 84
novel transcript, antisense to ADCYAP1
eukaryotic translation initiation factor 5A2 pseudogene

Top 10 values in Column "LNCAP":
SCYGR7      6.532796
KRTAP2-1    6.529890
KRTAP2-4    6.492452
KRTAP4-6    6.484903
KRTAP9-9    6.479307
KRTAP9-2    6.468585
KRTAP9-4    6.464421
KRTAP4-5    6.454317
KRTAP4-3    6.450180
KRTAP1-3    6.447542
Name: LNCAP, 

---
# System

In [22]:

# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(high_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
high_cat_oe_matrix = np.divide(high_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(high_cat_matrix), where=row_averages[:, np.newaxis]!=0)

print("High Cat matrix shape:")
print(high_cat_matrix.shape)
print(high_cat_matrix[0][:5])
print()
print("Average:", row_averages[0])
print()
print("High Cat OE matrix shape:")
print(high_cat_oe_matrix.shape)
print(high_cat_oe_matrix[0][:5])

High Cat matrix shape:
(57447, 9)
[  94.79237845   99.22251054  113.55603686  937.5426037  1645.39369115]

Average: 378.969252455521

High Cat OE matrix shape:
(57447, 9)
[0.25013211 0.26182206 0.29964446 2.47392789 4.34176039]


In [10]:
# from IPython.display import display

# df = pd.DataFrame(high_cat_oe_matrix, index=gene_names)
# df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
# "Immune System", "Integumentary System", "Muscular System", "Nervous System", 
# "Respiratory System", "Urogenital System"]
# display(df)

In [11]:
# loop through each column and get the top 10 highest values and their indexes

df = pd.DataFrame(high_cat_matrix, index=gene_names)
df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Cardiovascular System":
GAPDH     5.647761
TMSB4X    5.485484
AHNAK     5.433829
RPL41     5.426241
EEF1A1    4.878405
FTH1      4.813606
RPS18     4.804848
TPM1      4.730638
TMSB10    4.587425
RPL13A    4.565091
Name: Cardiovascular System, dtype: float64
Indexes for Column "Cardiovascular System":
Index(['GAPDH', 'TMSB4X', 'AHNAK', 'RPL41', 'EEF1A1', 'FTH1', 'RPS18', 'TPM1',
       'TMSB10', 'RPL13A'],
      dtype='object')

Top 10 values in Column "Connective Tissue":
RPS18     5.857983
RPL41     5.741125
FTL       5.589432
EEF1A1    5.426893
RPL13A    4.826380
RPS12     4.747020
FTH1      4.700924
RPS25     4.661744
RPL11     4.627384
RPL19     4.623405
Name: Connective Tissue, dtype: float64
Indexes for Column "Connective Tissue":
Index(['RPS18', 'RPL41', 'FTL', 'EEF1A1', 'RPL13A', 'RPS12', 'FTH1', 'RPS25',
       'RPL11', 'RPL19'],
      dtype='object')

Top 10 values in Column "Digestive System":
FTL                5.594203
RPL41              5.491354
G

In [12]:
df = pd.DataFrame(high_cat_oe_matrix, index=gene_names)
df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Cardiovascular System":
MYBPC3             8.470539
ENSG00000236958    8.351298
MYL7               8.236753
ENSG00000286800    8.219700
ENSG00000287891    8.186218
NPPA               8.147898
LINC02502          8.103888
ENSG00000286820    8.073964
TNNI3K             7.764016
ENSG00000286598    7.692011
Name: Cardiovascular System, dtype: float64

Gene descriptions for Column "Cardiovascular System":
myosin binding protein C3
novel transcript
myosin light chain 7
novel transcript
novel transcript
natriuretic peptide A
long intergenic non-protein coding RNA 2502
novel transcript
TNNI3 interacting kinase
novel transcript

Top 10 values in Column "Connective Tissue":
LINC02573          8.243832
ENSG00000287973    8.194014
ENSG00000237311    8.117703
RHAG               8.083788
ABCB10P4           8.006390
HBBP1              7.996683
HBZ                7.980091
GATA1              7.937128
ENSG00000264334    7.889233
LINC01029          7.878699
Name: Connective Tissue

In [23]:
high_cat_log2_oe_matrix = np.log2(high_cat_oe_matrix)
df = pd.DataFrame(high_cat_log2_oe_matrix, index=gene_names)
df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print()
    print('Gene descriptions for Column "{}":'.format(col))
    
    for gene in top_10.index:
        index = gene_names.index(gene)
        print(gene_description[index])
    print()


Top 10 values in Column "Cardiovascular System":
MYL2      3.164677
CSRP3     3.162553
TCAP      3.161837
LMOD2     3.161414
MYOZ2     3.160921
MYL3      3.160427
MYH7      3.160066
MYBPC3    3.158876
NRAP      3.157947
TNNT2     3.157807
Name: Cardiovascular System, dtype: float64

Gene descriptions for Column "Cardiovascular System":
myosin light chain 2
cysteine and glycine rich protein 3
titin-cap
leiomodin 2
myozenin 2
myosin light chain 3
myosin heavy chain 7
myosin binding protein C3
nebulin related anchoring protein
troponin T2, cardiac type

Top 10 values in Column "Connective Tissue":
MIR3926-2          3.169925
ENSG00000242327    3.169400
HBE1               3.117650
HBZP1              3.116678
LEP                3.114797
MIR3654            3.109950
ADIPOQ             3.108861
TRARG1             3.095794
LINCADL            3.091653
PLIN1              3.088465
Name: Connective Tissue, dtype: float64

Gene descriptions for Column "Connective Tissue":
microRNA 3926-2
ribosomal p

/tmp/ipykernel_7604/2203363747.py:1: RuntimeWarning: divide by zero encountered in log2
  high_cat_log2_oe_matrix = np.log2(high_cat_oe_matrix)
